In [1]:
import pandas as pd
import numpy as np

PROCESSED_FILE = "../data/processed/supervised_preprocessed.parquet"

df = pd.read_parquet(PROCESSED_FILE)

print("Shape:", df.shape)
print("Wells:", df["WELL"].nunique())

Shape: (1430974, 15)
Wells: 118


our defined model feature is called as baseline feature

In [4]:
MODEL_FEATURES = [
    "GR",
    "RDEP_LOG10",
    "RMED_LOG10",
    "DTC",
    "RHOB"
]

print(MODEL_FEATURES)

['GR', 'RDEP_LOG10', 'RMED_LOG10', 'DTC', 'RHOB']


In [5]:
baseline_features = df[MODEL_FEATURES].copy()

print("Baseline features:", baseline_features.columns.tolist())
print("Number of baseline features:", baseline_features.shape[1])

Baseline features: ['GR', 'RDEP_LOG10', 'RMED_LOG10', 'DTC', 'RHOB']
Number of baseline features: 5


A run is a consecutive sequence of samples having the same lithology.
Since the sampling interval is ~0.152 m, this tells us the typical thickness of labelled lithology intervals.


In [6]:

# Check consecutive lithology runs within each well
run_lengths = []

for well, g in df.groupby("WELL", sort=False):
    labels = g["FORCE_2020_LITHOFACIES_LITHOLOGY"].to_numpy()

    # Ignore NaN labels
    valid = ~pd.isna(labels)
    labels = labels[valid]

    if len(labels) == 0:
        continue

    # Find boundaries where lithology changes
    changes = np.r_[True, labels[1:] != labels[:-1], True]
    idx = np.flatnonzero(changes)

    lengths = np.diff(idx)

    run_lengths.extend(lengths)

run_lengths = np.asarray(run_lengths)

print("Number of lithology runs:", len(run_lengths))
print()
print("Run length statistics (samples):")
print(pd.Series(run_lengths).describe())

print()
print("Run length statistics (metres):")
run_m = run_lengths * 0.152
print(pd.Series(run_m).describe())

Number of lithology runs: 40199

Run length statistics (samples):
count    40199.000000
mean        35.597254
std        112.596617
min          1.000000
25%          5.000000
50%          9.000000
75%         25.000000
max       4353.000000
dtype: float64

Run length statistics (metres):
count    40199.000000
mean         5.410783
std         17.114686
min          0.152000
25%          0.760000
50%          1.368000
75%          3.800000
max        661.656000
dtype: float64


In [7]:
# Reconstruct missing values before calculating local features
feature_df = df.copy()

for feature in MODEL_FEATURES:
    missing_col = f"{feature}_MISSING"
    
    if missing_col in feature_df.columns:
        feature_df.loc[feature_df[missing_col] == 1, feature] = np.nan

print("Feature data prepared for rolling calculations.")

print(
    feature_df[MODEL_FEATURES]
    .isna()
    .mean()
    .mul(100)
    .round(2)
)

Feature data prepared for rolling calculations.
GR             0.09
RDEP_LOG10     0.77
RMED_LOG10     3.45
DTC            6.05
RHOB          13.14
dtype: float64


first engineered feature group: local rolling statistics.

In [8]:
ROLLING_WINDOWS = [5, 9, 21]

engineered_df = feature_df.copy()

for feature in MODEL_FEATURES:
    for window in ROLLING_WINDOWS:
        group = engineered_df.groupby("WELL", sort=False)[feature]

        engineered_df[f"{feature}_ROLLMEAN_{window}"] = (
            group.transform(
                lambda x: x.rolling(window=window, center=True, min_periods=3).mean()
            )
        )

        engineered_df[f"{feature}_ROLLSTD_{window}"] = (
            group.transform(
                lambda x: x.rolling(window=window, center=True, min_periods=3).std()
            )
        )

print("Original features:", len(MODEL_FEATURES))
print("Total columns:", engineered_df.shape[1])
print("New rolling features:", engineered_df.shape[1] - feature_df.shape[1])

Original features: 5
Total columns: 45
New rolling features: 30


In [9]:
rolling_features = [
    c for c in engineered_df.columns
    if "_ROLLMEAN_" in c or "_ROLLSTD_" in c
]

print("Rolling features:", len(rolling_features))

print("\nMissingness:")
print(
    engineered_df[rolling_features]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .head(15)
)

print("\nExample statistics:")
print(
    engineered_df[rolling_features]
    .describe()
    .T[["mean", "std", "min", "max"]]
    .head(10)
)

Rolling features: 30

Missingness:
RHOB_ROLLSTD_5           13.14
RHOB_ROLLMEAN_5          13.14
RHOB_ROLLSTD_9           13.11
RHOB_ROLLMEAN_9          13.11
RHOB_ROLLSTD_21          13.02
RHOB_ROLLMEAN_21         13.02
DTC_ROLLMEAN_5            6.05
DTC_ROLLSTD_5             6.05
DTC_ROLLSTD_9             6.03
DTC_ROLLMEAN_9            6.03
DTC_ROLLSTD_21            5.98
DTC_ROLLMEAN_21           5.98
RMED_LOG10_ROLLMEAN_5     3.45
RMED_LOG10_ROLLSTD_5      3.45
RMED_LOG10_ROLLSTD_9      3.39
dtype: float64

Example statistics:
                            mean        std       min         max
GR_ROLLMEAN_5          70.472952  33.849795  0.168961  961.742273
GR_ROLLSTD_5            2.854923   3.458944  0.000000  306.047296
GR_ROLLMEAN_9          70.472693  33.620567  0.378109  817.977715
GR_ROLLSTD_9            3.846626   4.548699  0.000000  360.120116
GR_ROLLMEAN_21         70.472564  33.172044  1.650216  637.204636
GR_ROLLSTD_21           5.295576   6.042336  0.000000  357.036005
RD

In [10]:
engineered_df["RESISTIVITY_SEPARATION"] = (
    engineered_df["RDEP_LOG10"] - engineered_df["RMED_LOG10"]
)

print(engineered_df["RESISTIVITY_SEPARATION"].describe())
print()
print(
    "Missing:",
    engineered_df["RESISTIVITY_SEPARATION"].isna().mean() * 100,
    "%"
)

count    1.380877e+06
mean    -1.674253e-02
std      1.534960e-01
min     -3.976862e+00
25%     -5.539525e-02
50%     -5.823534e-03
75%      3.233072e-02
max      3.860547e+00
Name: RESISTIVITY_SEPARATION, dtype: float64

Missing: 3.5009021827091193 %


Missing: 3.50%, close to the higher missingness of RMED_LOG10.

Median: about −0.0058, so the two logs are often fairly similar in log space.

The range is broad enough that it may capture useful geological/petrophysical variation.

In [11]:
BASELINE_FEATURES = MODEL_FEATURES.copy()

ROLLING_FEATURES = [
    c for c in engineered_df.columns
    if "_ROLLMEAN_" in c or "_ROLLSTD_" in c
]

ENGINEERED_FEATURES = [
    "RESISTIVITY_SEPARATION"
]

print("Baseline:", len(BASELINE_FEATURES))
print("Rolling:", len(ROLLING_FEATURES))
print("Other engineered:", len(ENGINEERED_FEATURES))
print("Total candidate:", len(
    BASELINE_FEATURES + ROLLING_FEATURES + ENGINEERED_FEATURES
))

Baseline: 5
Rolling: 30
Other engineered: 1
Total candidate: 36


In [12]:
MISSING_FEATURES = [
    f"{feature}_MISSING"
    for feature in MODEL_FEATURES
]

print("Missingness features:")
print(MISSING_FEATURES)

print("\nUnique values:")
print(engineered_df[MISSING_FEATURES].nunique())

print("\nMissingness rates:")
print(
    engineered_df[MISSING_FEATURES]
    .mean()
    .mul(100)
    .round(2)
)

Missingness features:
['GR_MISSING', 'RDEP_LOG10_MISSING', 'RMED_LOG10_MISSING', 'DTC_MISSING', 'RHOB_MISSING']

Unique values:
GR_MISSING            2
RDEP_LOG10_MISSING    2
RMED_LOG10_MISSING    2
DTC_MISSING           2
RHOB_MISSING          2
dtype: int64

Missingness rates:
GR_MISSING             0.09
RDEP_LOG10_MISSING     0.77
RMED_LOG10_MISSING     3.45
DTC_MISSING            6.05
RHOB_MISSING          13.14
dtype: float64


In [13]:
ENGINEERED_FILE = "../data/processed/engineered_features.parquet"

engineered_df.to_parquet(
    ENGINEERED_FILE,
    index=False
)

print("Saved:", ENGINEERED_FILE)
print("Shape:", engineered_df.shape)

Saved: ../data/processed/engineered_features.parquet
Shape: (1430974, 46)
